# Lilylet NotaGen — INT8 ONNX-Runtime autoregressive generation

Loads the INT8-quantized ONNX weights (`patch_int8.onnx` + `token_int8.onnx`, exported by `tools/export_lilylet_int8_ort.py`) and runs hierarchical patch/token autoregressive generation through ONNX Runtime instead of PyTorch.

The `ORTGenerator` (from `tests/bench_lilylet_int8_ort.py`) mirrors `LilyletPatchyGenerator.generate`: the two heavy transformer forwards run as ORT sessions, while the cheap embedding lookup / patch-state splice / sampling stay in numpy/torch. A torch `LilyletPatchyGenerator` is still built — only for the tokenizer and the patch_size / bos / eos ids.

In [1]:
from pathlib import Path
import sys
import os
import time

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'starry').is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# ORTGenerator lives in the tests/ bench module
if str(REPO_ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'tests' / 'lilylet'))

import torch
from starry.utils.config import Configuration
from starry.lilylet.patchyGenerator import LilyletPatchyGenerator
from bench_lilylet_int8_ort import ORTGenerator, ORTGeneratorKV

# The run whose int8 onnx we exported. RUN dir holds best.chkpt, .state.yaml and onnx/.
RUN = os.path.expanduser('~/data/models/deep-starry-logs/lilylet/20260611-lilylet-notagenx-1m0611-llama')
CKPT = os.path.join(RUN, 'best.chkpt')
ONNX_DIR = os.path.join(RUN, 'onnx')
PATCH_INT8 = os.path.join(ONNX_DIR, 'patch_int8.onnx')
PATCH_KV_INT8 = os.path.join(ONNX_DIR, 'patch_kv_int8.onnx')
TOKEN_INT8 = os.path.join(ONNX_DIR, 'token_int8.onnx')
TOKEN_KV_INT8 = os.path.join(ONNX_DIR, 'token_kv_int8.onnx')
THREADS = 14

TOKENIZER = str(REPO_ROOT / 'assets' / 'lilylet-tokenizer.json')
assert os.path.isfile(PATCH_INT8) and os.path.isfile(TOKEN_INT8), 'run tools/export_lilylet_int8_ort.py first'
print('run :', RUN)
print('int8:', os.path.getsize(PATCH_INT8)/1e6, 'MB +', os.path.getsize(TOKEN_INT8)/1e6, 'MB')

/home/camus/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


run : /home/camus/data/models/deep-starry-logs/lilylet/20260611-lilylet-notagenx-1m0611-llama
int8: 393.876554 MB + 117.037574 MB


In [2]:
# Build a torch generator from the run's config (architecture from .state.yaml).
# It loads the fp32 checkpoint too, but we only use it for the tokenizer + the
# patch_size / bos / eos / pad ids; all heavy forwards go through ORT below.
torch.set_num_threads(THREADS)
config = Configuration.createOrLoad(RUN, volatile=True)
gen = LilyletPatchyGenerator.from_config(config, CKPT, tokenizer_path=TOKENIZER, device='cpu')
print('base_type:', config['model.args.base_type'], '| patch_size:', gen.patch_size)
print('pad/bos/eos:', gen.pad_id, gen.bos_id, gen.eos_id)

# Wrap the int8 ONNX sessions. This is the actual inference engine.
ort_gen = ORTGenerator(gen, PATCH_INT8, TOKEN_INT8, threads=THREADS)
print('ORT int8 sessions ready')

base_type: llama | patch_size: 16
pad/bos/eos: 0 1 2


ORT int8 sessions ready


In [3]:
# Sanity: encode a protected token and decode it back through the tokenizer.
demo = gen.tokenizer.encode('[r:0/8]')
print('encode "[r:0/8]" ->', demo)
print('decode back        ->', repr(gen.patch_to_text(demo)))

encode "[r:0/8]" -> [91, 114, 58, 48, 47, 56, 93]
decode back        -> '[r:0/8]'


In [4]:
# Conditional generation: seed a metadata header and let the int8 model continue.
# (ORTGenerator.generate mirrors the torch generate signature, minus `verbose`.)
PROMPT = '[composer "Schubert, Franz"]\n[genre "Romantic"]\n[instrument "Keyboard"]\n'
torch.manual_seed(0)
t0 = time.perf_counter()
text = ort_gen.generate(prompt_text=PROMPT, max_patches=1024,
                        temperature=0.9, top_k=20, top_p=0.95,
                        measures=8, postprocess=True)
dt = time.perf_counter() - t0
print(text)
print('\n===== %d chars, %d lines in %.1fs =====' % (len(text), text.count('\n') + 1, dt))

[composer "Schubert, Franz"]
[genre "Romantic"]
[instrument "Keyboard"]

\staff "1" \key g \major \major \time 2/4 \clef "treble" \tempo 4=130 ^\markup "Allegro" b'16\pp( c \\
\staff "2" \clef "bass" r8 | % r:0/8

\staff "1" \key g \major \major \time 2/4 d'8 d)-. d-. g-. \\
\staff "2" <g b>8-. <g b>-. <g b>-. <g d'>-. | % r:1/7

\staff "1" \key g \major \time 2/4 g''4(-> d8 d)-. \\
\staff "2" <g b d>8-. <g b d>-. <g b d>-. <g b d>-. | % r:2/6

\staff "1" \key g \major \time 2/4 \grace e'8 d-. d16( e d8)-. c-. \\
\staff "2" <g b d>8-. <g b d>-. <g c d>-. <g c d>-. | % r:3/59

\staff "1" \key g \major \time 2/4 b'4( d8 d)-. \\
\staff "2" <g b d>8-. <g b d>-. <g b d>-. <g b d>-. | % r:4/72

\staff "1" \key g \major \time 2/4 <d' fs a>8\<-. <d fs a>-. <d fs a>-. <d fs a>-. \\
\staff "2" <d, fs a d>8-. <d fs a d>-. <d fs a d>-. <d fs a d>-. | % r:5/71

\staff "1" \key g \major \time 2/4 <d' g b>4\>(-> <b d>8 <g b>)\! \\
\staff "2" <g, b d g>8-. <g b d g>-. <g b d g>-. <g b d g>-. | % r:6/7

In [5]:
# Save the generated piece to a .lyl file.
out_dir = REPO_ROOT / 'tests' / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'lilylet_onnx_int8_generated.lyl'
out_path.write_text(text)
print('wrote', out_path)

wrote /home/camus/work/deep-starry/tests/output/lilylet_onnx_int8_generated.lyl


## KV cache: patch-level and token-level (`ORTGeneratorKV`)

`ORTGeneratorKV` keeps per-layer K/V caches so each decoder advances one step at a time (O(1)) instead of recomputing the whole sequence (O(T)):

- **patch-level** (`patch_kv_int8.onnx`): caches the patch sequence — the dominant cost for long pieces.
- **token-level** (`token_kv_int8.onnx`, optional via `token_kv_onnx=`): caches the ≤16 tokens inside each patch — a smaller, constant per-patch win.

**Equivalence:** in fp32 both KV decoders are numerically identical to full recompute (cos 1.0). In INT8 each graph is quantized independently, so per-step rounding differs slightly; under greedy decoding the patch-level difference feeds slightly different patch states and the sequence can branch (the same effect as torch-fp32 vs ORT-int8). Token-KV vs non-KV is byte-identical within a single patch. All outputs are valid Lilylet.

Below we run the **same seed** through baseline / patch-KV / patch+token-KV and compare output + wall-clock.

In [6]:
# Build the KV-cache generators (share the same torch gen + tokenizer).
#   ort_kv      : patch-level KV only  (token path = full recompute)
#   ort_kv_full : patch-level + token-level KV (both incremental)
assert os.path.isfile(PATCH_KV_INT8), 'patch_kv_int8.onnx missing; re-run tools/export_lilylet_int8_ort.py'
assert os.path.isfile(TOKEN_KV_INT8), 'token_kv_int8.onnx missing; re-run tools/export_lilylet_int8_ort.py'
ort_kv = ORTGeneratorKV(gen, PATCH_KV_INT8, TOKEN_INT8, threads=THREADS)
ort_kv_full = ORTGeneratorKV(gen, PATCH_KV_INT8, TOKEN_INT8, threads=THREADS, token_kv_onnx=TOKEN_KV_INT8)
print('patch_kv_int8:', os.path.getsize(PATCH_KV_INT8)/1e6, 'MB | token_kv_int8:', os.path.getsize(TOKEN_KV_INT8)/1e6, 'MB')
print('ORT KV-cache sessions ready (patch-only + patch+token)')

patch_kv_int8: 393.986221 MB | token_kv_int8: 117.080835 MB
ORT KV-cache sessions ready (patch-only + patch+token)


In [7]:
# Compare baseline (full recompute) vs patch-KV vs patch+token-KV, GREEDY + same
# seed so they are directly comparable. Greedy = temperature 1e-6, top_k 1.
SEED = 0
GEN_KW = dict(prompt_text=PROMPT, max_patches=512, temperature=1e-6, top_k=1, top_p=1.0,
              measures=16, postprocess=True)

def run (g):
    torch.manual_seed(SEED)
    t0 = time.perf_counter()
    txt = g.generate(**GEN_KW)
    return txt, time.perf_counter() - t0

text_base, dt_base = run(ort_gen)
text_kv, dt_kv = run(ort_kv)
text_kv_full, dt_kv_full = run(ort_kv_full)

print('%-26s %7s  %6s  %6s' % ('', 'time', 'chars', 'speedup'))
print('%-26s %6.1fs  %5d  %6s' % ('baseline (full recompute)', dt_base, len(text_base), '1.00x'))
print('%-26s %6.1fs  %5d  %5.2fx' % ('patch-KV', dt_kv, len(text_kv), dt_base / dt_kv))
print('%-26s %6.1fs  %5d  %5.2fx' % ('patch+token-KV', dt_kv_full, len(text_kv_full), dt_base / dt_kv_full))
print()
print('byte-identical (base vs patch-KV)     :', text_base == text_kv)
print('byte-identical (patch-KV vs full-KV)  :', text_kv == text_kv_full)
print('(int8 graphs quantize independently -> greedy may branch; all outputs are valid Lilylet)')

                              time   chars  speedup
baseline (full recompute)    18.2s   1790   1.00x
patch-KV                     12.6s   1910   1.45x
patch+token-KV                9.2s   1945   1.99x

byte-identical (base vs patch-KV)     : False
byte-identical (patch-KV vs full-KV)  : False
(int8 graphs quantize independently -> greedy may branch; all outputs are valid Lilylet)


In [8]:
# Save the three greedy outputs side by side for inspection / diff.
out_dir = REPO_ROOT / 'tests' / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
p_base = out_dir / 'lilylet_onnx_int8_greedy_base.lyl'
p_kv = out_dir / 'lilylet_onnx_int8_greedy_kv.lyl'
p_kv_full = out_dir / 'lilylet_onnx_int8_greedy_kv_full.lyl'
p_base.write_text(text_base)
p_kv.write_text(text_kv)
p_kv_full.write_text(text_kv_full)
print('wrote', p_base)
print('wrote', p_kv)
print('wrote', p_kv_full)
print('diff with:  diff', p_base, p_kv_full)

wrote /home/camus/work/deep-starry/tests/output/lilylet_onnx_int8_greedy_base.lyl
wrote /home/camus/work/deep-starry/tests/output/lilylet_onnx_int8_greedy_kv.lyl
wrote /home/camus/work/deep-starry/tests/output/lilylet_onnx_int8_greedy_kv_full.lyl
diff with:  diff /home/camus/work/deep-starry/tests/output/lilylet_onnx_int8_greedy_base.lyl /home/camus/work/deep-starry/tests/output/lilylet_onnx_int8_greedy_kv_full.lyl
